In [ ]:
import pyvisa
from sentio_prober_control.Communication.CommunicatorGpib import *
from sentio_prober_control.Sentio.ProberSentio import *
import numpy as np
import pandas as pd
from time import sleep
import glob

In [ ]:
# Load all the csvs generated and check weather there are files have missing or NaN values

# List to store tuples of (col, row, site) for CSV files with missing or overflow voltage entries
invalid_files = []

# Find all CSV files that match the naming pattern
csv_files = glob.glob("Sweep_Voltage_*.csv")

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    
    # Convert the "Voltage (V)" column to numeric; non-numeric values become NaN
    df["Voltage (V)"] = pd.to_numeric(df["Voltage (V)"], errors="coerce")
    
    # Check if any voltage is missing or greater than 1 volt
    if df["Voltage (V)"].isna().any() or (df["Voltage (V)"] > 1).any():
        # Extract the part of the filename after "Sweep_Voltage_" and before ".csv"
        name_part = csv_file[len("Sweep_Voltage_"):-len(".csv")]
        parts = name_part.split('_')
        # Expecting the parts: [col, row, site, ...]; extract the first three parameters if available
        if len(parts) >= 3:
            invalid_files.append((parts[0], parts[1], parts[2]))

# Print the results
print("CSV files with missing or overflow voltage entries (col, row, site):")
for item in invalid_files:
    print(item)




In [ ]:
class CustomCommunicatorGpib(CommunicatorGpib):
   def __init__(self, vendor, address):
       super().__init__(vendor)
       self.address = address
       self.rm = pyvisa.ResourceManager()
       self.instrument = self.rm.open_resource(self.address)
   def set_timeout(self, timeout_ms):
       self.instrument.timeout = timeout_ms
   def send(self, command):
       self.instrument.write(command)
   def read_line(self):
       return self.instrument.read()

In [ ]:
# Create the custom communicator with National Instruments card
comm = CustomCommunicatorGpib(GpibCardVendor.NationalInstruments, "GPIB0::13::INSTR")
# Set the timeout period (example: 10000 ms = 10 seconds)
comm.set_timeout(10000) # Timeout is set in milliseconds
# Initialize SentioProber with the communicator
prober = SentioProber(comm)
# Send an identification query to the instrument
comm.send("*IDN?")
response = comm.read_line()
print("Instrument ID: {0}".format(response))
# Create connection with SMU and test connection
rm =pyvisa.ResourceManager()
SMU = rm.open_resource('GPIB0::18::INSTR')
nVolt = rm.open_resource('GPIB0::6::INSTR')
# Reset the instrument and configure it for the IV sweep

# Configure the SMU as a current source
SMU.write('*RST')  # Reset the instrument
SMU.write('SOUR:FUNC CURR')  # Set the source function to current
SMU.write('SOUR:CURR:RANG 20E-6')  # Set the current range to 20 µA
SMU.write('SENS:FUNC "VOLT"')  # Set the sense function to voltage
SMU.write('SENS:VOLT:RANG 20E-3')  # Set the voltage range to 10 mV
SMU.write('SENS:VOLT:NPLC 1')
SMU.write('OUTP ON')  # Enable the outputv
# Configure the nanovoltmeter
nVolt.write('*RST')  # Reset the instrument
nVolt.write('CONF:VOLT:DC 10E-3')  # Set measurement function to DC voltage, 10 mV range

current_positive = 100e-6  # Positive current: 100 µA
current_negative = -100e-6  # Negative current: -100 µA
num_cycles = 10  # Number of cycles to alternate between currents

start_current = -100e-6  # Start current: -100 µA
stop_current = 100e-6  # Stop current: 100 µA
num_points = 20  # Number of sweep points
iv_sweep_currents = np.linspace(start_current, stop_current, num_points)

delta_sweep_currents = []
for i in range(100, 0, -5):
    delta_sweep_currents.extend([i, -i])
delta_sweep_currents.append(0)  # Add the 0 current at the end
delta_sweep_currents = [current * 1e-6 for current in delta_sweep_currents]



landing_count= 1

In [ ]:
# Move to the structure that needs to be measured
col,row,site = prober.map.step_die(1,0,81)
print(f'Moved to position {col}, {row} (Site: {site}).')
print("Please operate on system to make sure the probe tips are in the correct location")

In [ ]:
# Landing, test, and separation
prober.move_chuck_contact()
sleep(0.3)
data = []

for current in delta_sweep_currents:
    SMU.write(f'SOUR:CURR {current}')  # Set the current
    sleep(0.1)  # Wait for settling time
    voltage = nVolt.query('READ?')  # Read the voltage from the nanovoltmeter
    data.append((current, float(voltage)))  # Store the current and voltage
df = pd.DataFrame(data, columns=['Current (A)', 'Voltage (V)'])
filename = f'Sweep_Voltage_{col}_{row}_{site}_{l}.csv'
df.to_csv(filename, index=False)
print(f'Measurement done for position {col}, {row} (Site: {site}).')

prober.move_chuck_separation()
print("Please check the data")